# Privify — Fine-tuning Face Detector on WIDER FACE

This notebook fine-tunes a YOLOv8n model on a subset of the WIDER FACE
dataset (~5 000 images) to produce a face-specific detector.  The output
is a `best.pt` checkpoint that should be uploaded as a GitHub Release
asset (tagged `face-detector-v0.1`) for consumption by the main pipeline.

**Base model:** `yolov8n.pt` (COCO pre-trained)  
**Dataset:** WIDER FACE subset in YOLOv8 format (via Roboflow Universe)  
**Output:** `best.pt` — fine-tuned weights for single-class face detection

## 1. Setup

This notebook requires a **Colab T4 GPU** runtime.  
Expected total runtime: **~20–30 minutes** (depending on dataset size and
GPU availability).

In [ ]:
import os

REPO_DIR = "/content/privify"
REPO_URL = "https://github.com/jenz26/privify.git"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"{REPO_DIR} already exists, pulling latest changes...")
    !cd {REPO_DIR} && git pull --ff-only

%cd {REPO_DIR}

In [ ]:
import os
from pathlib import Path

MARKER = Path("/tmp/.privify_install_done")

if MARKER.exists():
    print("Dependencies already installed (post-restart). Ready to proceed.")
else:
    print("Installing dependencies (this takes 1-2 minutes)...")
    !pip install -q -r requirements.txt
    MARKER.touch()
    print("Done. Restarting kernel to load new dependencies cleanly...")
    print("After the restart, click 'Runtime → Run all' again.")
    os.kill(os.getpid(), 9)

In [ ]:
import sys
from pathlib import Path

REPO_DIR = "/content/privify"

# Ensure src/ is importable from the repo root.
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.training import TrainingConfig, train_face_detector

## 2. Download dataset

The dataset (Face Detection, ~2 500 images in YOLOv8 format) is hosted as
an asset of the `dataset-v0.1` release of this repository.  It was
originally sourced from Roboflow Universe and redistributed here under
CC BY 4.0 for reproducibility without external dependencies.

In [ ]:
DATASET_URL = "https://github.com/jenz26/privify/releases/download/dataset-v0.1/widerface.zip"
DATASET_DIR = Path("/content/widerface")
DATA_YAML = DATASET_DIR / "data.yaml"

if DATA_YAML.exists():
    print(f"Dataset already at {DATASET_DIR}, skipping download.")
else:
    print("Downloading dataset from GitHub Releases...")
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    !curl -L "{DATASET_URL}" -o {DATASET_DIR}/dataset.zip
    print("Extracting...")
    !cd {DATASET_DIR} && unzip -q dataset.zip && rm dataset.zip
    if not DATA_YAML.exists():
        raise FileNotFoundError(
            f"Expected {DATA_YAML} after extraction. The archive layout may "
            "have changed; verify the Release asset."
        )
    print(f"Done. Dataset ready at {DATASET_DIR}")

## 3. Configure and launch training

Training runs for 50 epochs on the T4 GPU.  Adjust `epochs` or
`batch_size` if you run into memory issues or want faster iteration.

In [ ]:
config = TrainingConfig(
    dataset_yaml_path=DATASET_DIR / "data.yaml",
    output_dir=Path("/content/runs/face"),
    base_model="yolov8n.pt",
    epochs=50,
    image_size=640,
    batch_size=16,
    device="cuda",
)
result = train_face_detector(config)

print(f"\nTraining completed in {result.elapsed_seconds / 60:.1f} minutes")
print(f"Best weights: {result.best_weights_path}")
print("Final metrics:")
for key, value in result.final_metrics.items():
    print(f"  {key}: {value:.4f}")

## 4. Download trained weights

Download `best.pt` to your local machine, then upload it as a
GitHub Release asset:

```bash
gh release create face-detector-v0.1 best.pt \
    --title "Face detector v0.1" \
    --notes "YOLOv8n fine-tuned on WIDER FACE subset (~5k images, 50 epochs)"
```

In [ ]:
from google.colab import files

files.download(str(result.best_weights_path))

## Next steps

- Upload `best.pt` as a GitHub Release asset tagged `face-detector-v0.1`
- Update the `Detector` class to download weights from the release URL
  on first use (lazy download + local cache)
- Test the updated pipeline on the demo notebook with face-specific
  detection instead of the COCO "person" proxy
- Repeat the same process for license plates with the CCPD dataset